# D11 独立练习：分组与聚合

先读同目录 README.md；本文件用于保存你自己的代码与输出。
每次运行前，先在注释中写出预计的形状或数值。
需要参考时，另开 reference.ipynb 对照相同编号的步骤。

内核选择 Python (D Learning)。代码从上到下运行；Ctrl+S 保存。
本次读取 D:/Learning/study/d11/data/prices.csv，不用 D10 的清洗结果。


## 1. 读取数据

导入 Path 与 pandas，设置 D11 的 base_dir、data_path、output_dir；读取 df，查看全部记录与 shape。预计 (8, 5)，编号 105 的价格缺失。


In [7]:
# 预计形状：
# 在这里写导入、路径、read_csv 和查看代码。
from pathlib import Path  # Path：表示和拼接文件路径
import pandas as pd

base_dir = Path("D:/Learning/study/d11")  # 今天的学习文件夹
data_path = base_dir / "data" / "prices.csv"  # 拼接样本文件路径

df = pd.read_csv(data_path)  # 读取 CSV，默认把空字段识别为缺失值


print(df)
print(df.shape)


   record_id        date asset  price  volume
0        101  2026-10-12     A  100.0    1000
1        102  2026-10-12     B  200.0     800
2        103  2026-10-13     A  105.0    1200
3        104  2026-10-13     B  198.0     900
4        105  2026-10-14     A    NaN    1100
5        106  2026-10-14     B  205.0     850
6        107  2026-10-15     A  108.0    1300
7        108  2026-10-15     B  203.0     950
(8, 5)


## 2. 确认类型与缺失

将 price 转为数值、date 转为日期；运行 info 和 isna().sum()。预计价格缺失 1 个，其余列缺失 0 个。不要删掉价格缺失行。


In [8]:

df["price"] = pd.to_numeric(
    df["price"],   # 需要转换的价格列
    errors = "coerce",  # 无法转换为数字的内容变成缺失值
)

df["date"] = pd.to_datetime(
    df["date"],# 需要转换的日期列
    format = "%Y-%m-%d",# 日期格式：年-月-日
    errors = "coerce",# 非法日期变成 NaT，即日期缺失值
)
print(df.isna().sum())
df.info()  # 查看列的类型和非空数量

record_id    0
date         0
asset        0
price        1
volume       0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   record_id  8 non-null      int64         
 1   date       8 non-null      datetime64[ns]
 2   asset      8 non-null      object        
 3   price      7 non-null      float64       
 4   volume     8 non-null      int64         
dtypes: datetime64[ns](1), float64(1), int64(2), object(1)
memory usage: 448.0+ bytes


## 3. 分组并取出 A

按 asset 分组，查看 ngroups，再 get_group 取 A。预计两组；A 为 (4, 5)，价格为 100、105、缺失、108。


In [10]:
# 预计组数和 A 组形状：
# 在这里写 groupby、ngroups 和 get_group。
grouped = df.groupby("asset")
a_records = grouped.get_group("A")

print(a_records)
print(a_records.shape)

record_counts = grouped.size()                    # 每组有多少行，包含价格缺失行
valid_price_counts = grouped["price"].count()     # 每组价格列有多少个非空值   ##区别在于是否计入非空数值

print(record_counts)
print(valid_price_counts)

   record_id       date asset  price  volume
0        101 2026-10-12     A  100.0    1000
2        103 2026-10-13     A  105.0    1200
4        105 2026-10-14     A    NaN    1100
6        107 2026-10-15     A  108.0    1300
(4, 5)
asset
A    4
B    4
dtype: int64
asset
A    3
B    4
Name: price, dtype: int64


## 4. 分别计算六项统计

分别求记录数、非空价格数、最早日期、最晚日期、最高价、最低价。先写出 A 的六项预测，再运行。


In [12]:
# A 的记录数：
# A 的非空价格数：
# A 的最早、最晚日期：
# A 的最高、最低价格：
# 在这里分别计算并输出六项统计。    用 agg 生成完整汇总表

record_counts = grouped.size()  # 统计每个资产组的记录行数，含空价格行
valid_price_counts = grouped["price"].count()  # 统计每组非空价格数
start_dates = grouped["date"].min()  # 每组最早日期；不需要先按日期排序
end_dates = grouped["date"].max()  # 每组最晚日期
highest_prices = grouped["price"].max()  # 每组最高价，忽略价格缺失值
lowest_prices = grouped["price"].min()  # 每组最低价，忽略价格缺失值

print("记录数：", record_counts)  # A 为 4，B 为 4
print("非空价格数：", valid_price_counts)  # A 为 3，B 为 4
print("起始日期：", start_dates)  # A、B 都是 2026-10-12
print("结束日期：", end_dates)  # A、B 都是 2026-10-15
print("最高价：", highest_prices)  # A 为 108，B 为 205
print("最低价：", lowest_prices)  # A 为 100，B 为 198

记录数： asset
A    4
B    4
dtype: int64
非空价格数： asset
A    3
B    4
Name: price, dtype: int64
起始日期： asset
A   2026-10-12
B   2026-10-12
Name: date, dtype: datetime64[ns]
结束日期： asset
A   2026-10-15
B   2026-10-15
Name: date, dtype: datetime64[ns]
最高价： asset
A    108.0
B    205.0
Name: price, dtype: float64
最低价： asset
A    100.0
B    198.0
Name: price, dtype: float64


## 5. 用 agg 整理汇总表

按资产分组，设 as_index=False，使用命名聚合创建 summary。列名为 asset、record_count、valid_price_count、start_date、end_date、highest_price、lowest_price。预计 (2, 7)。


In [20]:
# 预计汇总表形状：
# 在这里独立写出 groupby 和 agg。
summary = df.groupby(
    "asset",
    as_index = False,   # 让 asset 保持为普通列
).agg(
    record_count=("price","size"),
    valid_price_count = ("price","count"),
    start_date = ("date","min"),
    end_date = ("date","max"),
    highest_price = ("price","max"),
    lowest_price =("price","min"),
)

print(summary)
print(summary.shape)

  asset  record_count  valid_price_count start_date   end_date  highest_price  \
0     A             4                  3 2026-10-12 2026-10-15          108.0   
1     B             4                  4 2026-10-12 2026-10-15          205.0   

   lowest_price  
0         100.0  
1         198.0  
(2, 7)


## 6. 手算 A 并核对

先按纸上的结果写 manual_a，再把 summary 临时设为 asset 索引，取 A 的统计逐项比较。解释为什么缺失价格不影响 A 的记录数。


In [16]:
# 先写纸上独立得到的六个预期值。
# 再读取程序统计结果，逐项核对。
# 用一句话解释 size 与 count 的区别：.size()包括非空元素 直接统计所有元素，.count()只记录非空元素

## 7. 保存汇总表

创建 outputs，导出 asset_summary.csv，不保存额外索引，日期格式 YYYY-MM-DD。核对记录数合计 8、非空价格数合计 7；df 仍有八行。


In [ ]:
# 在这里创建输出目录并保存 CSV。
# 记录数合计应该是多少？
# 非空价格数合计应该是多少？
# 原表 df 现在还有多少行？
output_dir = base_dir / "outputs"  # 指定结果保存文件夹

output_dir.mkdir(
    parents=True,    # 上级文件夹不存在时一起创建
    exist_ok=True,   # 文件夹已经存在时继续运行
)

summary.to_csv(
    output_dir / "asset_summary.csv",  # 指定结果文件路径
    index=False,                       # 不额外保存默认数字索引
    encoding="utf-8-sig",               # 使用方便 Windows 识别的编码
    date_format="%Y-%m-%d",             # 日期保存为年-月-日
)

print(summary["record_count"].sum())  # 总记录数应为 8
print(summary["valid_price_count"].sum())  # 非空价格数合计应为 7
print(len(df))  # 原表仍应有 8 行

## 8. 不看答案重写

在下方空白单元格重写完整的汇总过程，再回答 notes.md 中的问题。能运行不等于能独立写；主线完成后再对照参考。


In [ ]:
# 从读取数据、类型转换，到 groupby、agg 和保存，独立写一遍。


## 选做：计算缺失价格数

复制 summary，新建 missing_price_count=record_count-valid_price_count。预计 A 为 1、B 为 0。主汇总仍保持七列。


In [ ]:
# 在 summary.copy() 的副本上完成，不修改主汇总表。
